## Imports

In [2]:
import cobra
from cobra.io import read_sbml_model, write_sbml_model
from cobra import Metabolite, Reaction
import os
import pandas as pd

## Paths

In [3]:
# Path to your models
model_dir = '/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1_mdr_rdr_dp/'

save_dir = '/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1_mdr_rdr_dp_mb2/'

## Fix imbalanced reactions

### Functions

In [4]:
def change_metabolite_formula(metabolite, new_formula):
    model.metabolites.get_by_id(metabolite).formula = str(new_formula)

In [5]:
def change_metabolite_charge(metabolite, new_charge):
    model.metabolites.get_by_id(metabolite).charge = int(new_charge)

In [6]:
def replace_metabolite_in_reaction(reaction, old_metabolite, new_metabolite):
    reaction_model = model.reactions.get_by_id(reaction)
    try:
        old_met_model = model.metabolites.get_by_id(old_metabolite)
    except KeyError:
        return
    try:
        new_met_model = model.metabolites.get_by_id(new_metabolite)
    except KeyError:
        new_met_model = new_metabolite
    if old_met_model in reaction_model.metabolites:
        coefficient = reaction_model.get_coefficient(old_met_model)
        reaction_model.add_metabolites({new_met_model: coefficient})
        reaction_model.add_metabolites({old_met_model: -coefficient})
    else:
        return

In [7]:
def create_gpr_list(reaction):

    # Get the GPR as string
    gpr_string = reaction.gene_reaction_rule

    # If no GPR is given set ["None"] as the value in the output dict
    if gpr_string == '':
        return ['']

    # If GPR are found formate them into a list by first splitting them by the 'or' relation
    # Genes with an 'and' relation will be split further and added to the GPR list as an list themself
    else:
        gpr_list_or_split = list(gpr_string.split(' or '))

        # Create a list where both 'or' and 'and' relations are kept 
        gpr_list_or_and_split = []

        # Check if an entry in the list contains an 'and'
        for entry in gpr_list_or_split:

            # If yes the split this entry into a new list, clean it and add the list to the value list
            if 'and' in entry:
                new_entry = list(entry.split(' and '))
                new_entry_cleaned = [s.replace("(", "").replace(")", "") for s in new_entry]

                # Sort this sublist alphabetically
                gpr_list_or_and_split.append(sorted(new_entry_cleaned))

            # If no then just append the entry into the output list
            else:
                gpr_list_or_and_split.append(entry)

        # Sort the output list alphabetically by treating each entry as a string 
        # Add the sorted value list to the output dict
        output = sorted(gpr_list_or_and_split, key=lambda x: ' '.join(x) if isinstance(x, list) else x)

        return output

In [8]:
# A function to transform a GPR list back into a GPR string 

def create_gpr_strings(gpr_list):
    result = []
    for item in gpr_list:
        if isinstance(item, list):
            # Join sublist with ' and ' and wrap in parentheses
            sub_str = ' and '.join(item)
            result.append(f'({sub_str})')
        else:
            result.append(item)
    return ' or '.join(result)

In [9]:
def remove_duplicates_in_mixed_list(list):

    # To keep track of checked list items
    seen = set()

    # Output list with duplicates filtered out
    unique_items = []

    # Check each item and treat it depending on its type
    for item in list:
        # Try to convert to tuple (for lists), otherwise leave as-is (for strings)
        try:
            key = tuple(item)
        except TypeError:
            key = item

        # Check if item was alredy checked and append it if not
        if key not in seen:
            seen.add(key)
            unique_items.append(item)

    return unique_items

In [10]:
# Inputs are two reaction ids
def replace_reaction(old_reaction, new_reaction):

    try:
        old_reaction_model = model.reactions.get_by_id(old_reaction)
    except KeyError:
        return 'no old'
    try:
        new_reaction_model = model.reactions.get_by_id(new_reaction)
    except KeyError:
        old_reaction_model.id = new_reaction
        return 'no new'
    
    old_reaction_gpr_list = create_gpr_list(old_reaction_model)
    new_reaction_gpr_list = create_gpr_list(new_reaction_model)

    extended_list = old_reaction_gpr_list + new_reaction_gpr_list

    unique_lists = remove_duplicates_in_mixed_list(extended_list)

    combined_reaction_gpr_string = create_gpr_strings(unique_lists)

    new_reaction_model.gene_reaction_rule = combined_reaction_gpr_string

    model.remove_reactions(old_reaction_model)

    return 'replaced'

In [11]:
def fix_proton_imbalance(reaction):
    balance_check = model.reactions.get_by_id(reaction).check_mass_balance()
    if 'H' in balance_check.keys():
        proton_amount = balance_check['H']
        model.reactions.get_by_id(reaction).add_metabolites({model.metabolites.get_by_id('h_c'): -proton_amount})
    else:
        return

In [ ]:
def fix_special_case_reaction(reaction):
    model_reaction = model.reactions.get_by_id(reaction)

    if reaction == 'MECDPDH3_syn' or 'MECDPDH4E':
        try:
            model_reaction.add_metabolites({
                model.metabolites.get_by_id('fdxox_c'): 2.0,
                model.metabolites.get_by_id('h2mb4p_c'): 1.0,
                model.metabolites.get_by_id('h2o_c'): 1.0
            })
        except KeyError:
            model_reaction.add_metabolites({
                fdxox_c: 2.0,
                model.metabolites.get_by_id('h2mb4p_c'): 1.0,
                model.metabolites.get_by_id('h2o_c'): 1.0
            })
        model_reaction.bounds = (1000.0, 1000.0)
    
    if reaction == 'GLYCS_I' or 'GLYCS_II':
        model.remove_reactions([model_reaction])

    if reaction == 'FNOR':
        try:
            model_reaction.add_metabolites({
                model.metabolites.get_by_id('fdxox_c'): 2.0,
                model.metabolites.get_by_id('nadph_c'): 1.0,
            })
        except KeyError:
            model_reaction.add_metabolites({
                fdxox_c: 2.0,
                model.metabolites.get_by_id('nadph_c'): 1.0,
            })

    if reaction == 'THZSN_1':
        try:
            model_reaction.add_metabolites({
                model.metabolites.get_by_id('fdxox_c'): -1.0,
                model.metabolites.get_by_id('tyr__L_c'): -1.0,
                model.metabolites.get_by_id('4hba_c'): 1.0,
                model.metabolites.get_by_id('4mhetz_c'): 1.0,
                model.metabolites.get_by_id('co2_c'): 1.0,
                model.metabolites.get_by_id('fdxrd_c'): 1.0,
                model.metabolites.get_by_id('h2o_c'): 1.0,
                model.metabolites.get_by_id('h_c'): 2.0,
                model.metabolites.get_by_id('nh4_c'): 1.0,
                model.metabolites.get_by_id('pyr_c'): 1.0
            })
        except KeyError:
            model_reaction.add_metabolites({
                fdxox_c: -1.0,
                model.metabolites.get_by_id('tyr__L_c'): -1.0,
                model.metabolites.get_by_id('4hba_c'): 1.0,
                model.metabolites.get_by_id('4mhetz_c'): 1.0,
                model.metabolites.get_by_id('co2_c'): 1.0,
                model.metabolites.get_by_id('fdxrd_c'): 1.0,
                model.metabolites.get_by_id('h2o_c'): 1.0,
                model.metabolites.get_by_id('h_c'): 2.0,
                model.metabolites.get_by_id('nh4_c'): 1.0,
                model.metabolites.get_by_id('pyr_c'): 1.0
            })

### Things to change

In [71]:
metabolites_to_change_formula = {
    'fpram_c': 'C8H14N3O8P',
    'aad_c': 'C12H14N5O8P',
    'mcbtt_c': 'C43H71N5O10',
    'phdcaACP_c': 'C407H639N96O144PS3',
    'prepphthACP_c': 'C419H661N96O147S3P',
    'arachACP_c': 'C404H641N96O143PS3',
    'prephthACP_c': 'C416H663N96O146PS3',
    '5ohhip_c': 'C13H19O4',
    'ficytb_c': 'C33H32FeN4O4',
    '3hcmrs7eACP_c': 'C398H627N96O144PS3',
    '3hbutACP_c': 'C388H609O144N96PS3',
    '23dhbzs3_e': 'C30H29N3O16',
    'nwharg_c': 'C6H14N4O3',
    '2mb_p_c': 'C5H9O5P',
    'selhcys_c': 'C4H9NO2Se',
    'ib_p_c': 'C4H7O5P',
    'istfrnB_e': 'C19FeH23N2O19',
    'stfrnB_c': 'C19H23N2O19',
    "hip_c": "C13H17O4",
    "ficytb_c": "C34H32FeN4O4"
}

In [72]:
metabolites_to_change_charge = {
    'prephth_c': 1,
    'pser_c': -2,
    '5ohhip_c': -1,
    'ficytb_c': 3,
    '3hbutACP_c': -1,
    'fe3dhbzs3_e': 3,
    '2mb_p_c': -2,
    '3opcoa_c': -4,
    'fe3mcbtt_c': 3,
    'ib_p_c': -2,
    'fpram_c': -2
}

In [73]:
reactions_change_protons = ['PENAM',
 'GLUFT',
 'UNK2',
 'THZSN',
 'OCDOR',
 'PHACA',
 'FRDO',
 'FMNAT_1',
 'RBFK_1',
 'GAPD_1',
 'GAPDy',
 'SHCHF_2',
 'ARD_1',
 'PSP_1',
 'IPDAB',
 'GLUFT',
 'NFORGLUAH',
 'CITDAPPS',
 'FRDO',
 'DHBZS3H',
 'FE3DHBZS3R',
 'FEDHBZS3R1',
 'FEDHBZS3R2',
 'ENTERH',
 'FEDHBZS3R3',
 'PRAIS',
 'GLUTRS_2',
 'BTS_1',
 'HEMEOS_1',
 'MI3PP_1',
 'PLPS_1',
 'MAN6Gpts',
 'TAGabc',
 'UACCpts',
 'UNK3',
 'HIBHr',
 'SEAHCYSHYD_1',
 'UPP3MT_2',
 'ECOAH2c',
 'PRFGS']

In [74]:
reactions_replace_fdxo_2_2_c = [
    'CO2FO',
    'THZSN_1',
    'NAR_syn',
    'MRF',
    'NOR_syn_1',
    'NAR_syn_2',
    'FNRR3',
    'MECDPDH3_syn',
    'MECDPDH4E',
    'FNOR',
    'SULR_syn',
    'VOR2bE',
    'OBFOX'
]

In [75]:
fdxox_c = Metabolite(
    'fdxox_c',
    formula='Fe2S2',
    charge=2,
    name='Oxidized ferredoxin',
    compartment='c')

In [76]:
metabolites_to_delete = [
    'ival_c'
]

In [77]:
reactions_to_delete = [
    'IPPT',
    'IVALt',
    'EX_ival_e',
    'SEAHCYSHYD_1'
]

In [78]:
models_with_special_case_reactions = {
    'Leaf59_or_mb1_mdr_rdr_dp.xml': ['MECDPDH3_syn'],
    'Leaf61_or_mb1_mdr_rdr_dp.xml': ['MECDPDH3_syn', 'MECDPDH4E', 'FNOR'],
    'Leaf414_or_mb1_mdr_rdr_dp.xml': ['MECDPDH3_syn', 'MECDPDH4E'],
    'Leaf126_or_mb1_mdr_rdr_dp.xml': ['MECDPDH3_syn', 'MECDPDH4E', 'FNOR'],
    'Leaf459_or_mb1_mdr_rdr_dp.xml': ['MECDPDH3_syn', 'MECDPDH4E'],
    'Leaf416_or_mb1_mdr_rdr_dp.xml': ['MECDPDH3_syn', 'MECDPDH4E'],
    'Leaf139_or_mb1_mdr_rdr_dp.xml': ['MECDPDH3_syn', 'MECDPDH4E', 'FNOR'],
    'Root189_or_mb1_mdr_rdr_dp.xml': ['MECDPDH3_syn', 'MECDPDH4E', 'FNOR'],
    'Leaf408_or_mb1_mdr_rdr_dp.xml': ['MECDPDH3_syn', 'MECDPDH4E'],
    'Root52_or_mb1_mdr_rdr_dp.xml': ['FNOR'],
    'Leaf53_or_mb1_mdr_rdr_dp.xml': ['FNOR', 'THZSN_1'],
    'Leaf50_or_mb1_mdr_rdr_dp.xml': ['FNOR', 'THZSN_1'],
    'Root227_or_mb1_mdr_rdr_dp.xml': ['GLYCS_I', 'GLYCS_II'],
    'Leaf436_or_mb1_mdr_rdr_dp.xml': ['GLYCS_I', 'GLYCS_II'],
    'Leaf159_or_mb1_mdr_rdr_dp.xml': ['GLYCS_I', 'GLYCS_II'],
    'Leaf161_or_mb1_mdr_rdr_dp.xml': ['GLYCS_I', 'GLYCS_II'],
    'Leaf320_or_mb1_mdr_rdr_dp.xml': ['GLYCS_I', 'GLYCS_II'],
    'Leaf151_or_mb1_mdr_rdr_dp.xml': ['GLYCS_I', 'GLYCS_II'],
    'Root553_or_mb1_mdr_rdr_dp.xml': ['GLYCS_I', 'GLYCS_II'],
    'Leaf183_or_mb1_mdr_rdr_dp.xml': ['GLYCS_I', 'GLYCS_II']
}

In [79]:
special_case_reactions = ["MECDPDH3_syn", "MECDPDH4E", "GLYCS_I", "GLYCS_II", "FNOR", "THZSN_1"]

### Main

In [83]:
for file in os.listdir(model_dir):
    print(file)
    if f'{file[:-4]}_mb2.xml' not in os.listdir(save_dir):
        model = read_sbml_model(model_dir+file)

        for met, formula in metabolites_to_change_formula.items():
            if f'{met[:-2]}_c' in model.metabolites:
                change_metabolite_formula(f'{met[:-2]}_c', formula)
            if f'{met[:-2]}_p' in model.metabolites:
                change_metabolite_formula(f'{met[:-2]}_p', formula)
            if f'{met[:-2]}_e' in model.metabolites:
                change_metabolite_formula(f'{met[:-2]}_e', formula)

        for met, charge in metabolites_to_change_charge.items():
            if f'{met[:-2]}_c' in model.metabolites:
                change_metabolite_charge(f'{met[:-2]}_c', charge)
            if f'{met[:-2]}_p' in model.metabolites:
                change_metabolite_charge(f'{met[:-2]}_p', charge)
            if f'{met[:-2]}_e' in model.metabolites:
                change_metabolite_charge(f'{met[:-2]}_e', charge)

        replace_reaction('PRFGS_1', 'PRFGS')

        replace_reaction('PRAIS_1', 'PRAIS')

        overwrite_reaction(model, "PSP", {"h2o_c": -1.0, "pser__L_c": -1.0, "pi_c": 1.0, "ser__L_c": 1.0}) #was wrong reaction before
        overwrite_reaction(model, "PRFGCL", {"atp_c": -1.0, "fpram_c": -1.0, "adp_c": 1.0, "air_c": 1.0, "h_c": 1.0, "pi_c": 1.0}) 
        overwrite_reaction(model, "HIPA", {"h_c": -1.0, "nadh_c": -1.0, "hip_c": -1.0, "nad_c": 1.0, "5ohhip_c": 1.0}) 

        '''if 'CITDAPPS' in model.reactions:
            model.reactions.get_by_id('CITDAPPS').add_metabolites({
                model.metabolites.get_by_id('h2o_c'): -1
                })
        '''
        for reaction in reactions_change_protons:
            if reaction in model.reactions:
                fix_proton_imbalance(reaction)

        for reaction in reactions_replace_fdxo_2_2_c:
            if reaction in model.reactions:
                if model.id in models_with_fdxo_2_2_c_but_no_fdxox_c:
                    replace_metabolite_in_reaction(reaction, 'fdxo_2_2_c', fdxox_c)
                else:
                    replace_metabolite_in_reaction(reaction, 'fdxo_2_2_c', 'fdxox_c')

        for met in metabolites_to_delete:
            if met in model.metabolites:
                met_model = model.metabolites.get_by_id(met)
                reactions = model.metabolites.get_by_id(met).reactions
                model.remove_reactions(reactions)
                model.remove_metabolites(met_model)

            for rxn in model.reactions:
                if rxn in special_case_reactions:
                    fix_special_case_reaction(rxn)
            
        write_sbml_model(model, save_dir+f'{file[:-4]}_mb2.xml')

262_or_mb1_mdr_rdr_dp.xml
397_or_mb1_mdr_rdr_dp.xml
1334_or_mb1_mdr_rdr_dp.xml
428_or_mb1_mdr_rdr_dp.xml
946_or_mb1_mdr_rdr_dp.xml
1208_or_mb1_mdr_rdr_dp.xml
2872_or_mb1_mdr_rdr_dp.xml
161_or_mb1_mdr_rdr_dp.xml
1362_or_mb1_mdr_rdr_dp.xml
1101_or_mb1_mdr_rdr_dp.xml
709_or_mb1_mdr_rdr_dp.xml
230_or_mb1_mdr_rdr_dp.xml
163_or_mb1_mdr_rdr_dp.xml
1234_or_mb1_mdr_rdr_dp.xml
2862_or_mb1_mdr_rdr_dp.xml
638_or_mb1_mdr_rdr_dp.xml
1174_or_mb1_mdr_rdr_dp.xml
868_or_mb1_mdr_rdr_dp.xml
1167_or_mb1_mdr_rdr_dp.xml
1391_or_mb1_mdr_rdr_dp.xml
939_or_mb1_mdr_rdr_dp.xml
1432_or_mb1_mdr_rdr_dp.xml
2774_or_mb1_mdr_rdr_dp.xml
1018_or_mb1_mdr_rdr_dp.xml
1124_or_mb1_mdr_rdr_dp.xml
793_or_mb1_mdr_rdr_dp.xml
895_or_mb1_mdr_rdr_dp.xml
2751_or_mb1_mdr_rdr_dp.xml
1338_or_mb1_mdr_rdr_dp.xml
644_or_mb1_mdr_rdr_dp.xml
1114_or_mb1_mdr_rdr_dp.xml
1357_or_mb1_mdr_rdr_dp.xml
364_or_mb1_mdr_rdr_dp.xml
1350_or_mb1_mdr_rdr_dp.xml
761_or_mb1_mdr_rdr_dp.xml
352_or_mb1_mdr_rdr_dp.xml
947_or_mb1_mdr_rdr_dp.xml
504_or_mb1_mdr_rdr_

# assess

In [81]:
def analyze_directory_imbalances(model_dir, output_csv="mass_imbalances_round2_2nditer.csv"):
    """
    Loads all SBML (.xml) models in model_dir, accumulates all reactions 
    with elemental mass imbalances, and saves them to a CSV file.
    """
    all_imbalances = []
    
    # List all XML files in the directory
    model_files = [f for f in os.listdir(model_dir) if f.endswith('.xml')]
    
    if not model_files:
        print(f"No .xml files found in {model_dir}")
        return pd.DataFrame()

    for file_name in model_files:
        file_path = os.path.join(model_dir, file_name)
        print(f"Processing {file_name}...")
        
        try:
            # Load the model
            model = read_sbml_model(file_path)
        except Exception as e:
            print(f"Error reading {file_name}: {e}")
            continue

        for rxn in model.reactions:
            # Skip boundary reactions (exchange, demand, sink) and growth 
            if rxn.boundary:
                continue
            if "Growth" in rxn.id:
                continue
                
            balance_errors = rxn.check_mass_balance()
            
            if balance_errors:
                # Isolate mass (elemental) imbalance by filtering out 'charge'
                mass_elements = {k: v for k, v in balance_errors.items() if k != 'charge'}
                
                # If mass_elements is not empty, we have a mass imbalance
                if mass_elements:
                    all_imbalances.append({
                        'Model_File': file_name,
                        'Model_ID': model.id,
                        'Reaction_ID': rxn.id,
                        'Reaction_Name': rxn.name,
                        'Reaction_String': rxn.reaction,
                        'Mass_Imbalance': str(mass_elements),
                        'Charge_Imbalance': balance_errors.get('charge', 0)
                    })

    # Convert to DataFrame
    df = pd.DataFrame(all_imbalances)
    
    # Save to file if data was found
    if not df.empty:
        df.to_csv(output_csv, index=False)
        print(f"\nSuccess! Found {len(df)} mass-imbalanced reactions across your models.")
        print(f"Saved results to '{output_csv}'")
    else:
        print("\nNo mass-imbalanced reactions found in any of the models.")
        
    return df


In [82]:
df_imbalances = analyze_directory_imbalances(save_dir)

Processing 2872_or_mb1_mdr_rdr_dp_mb2.xml...
Processing 1357_or_mb1_mdr_rdr_dp_mb2.xml...
Processing 397_or_mb1_mdr_rdr_dp_mb2.xml...
Processing 796_or_mb1_mdr_rdr_dp_mb2.xml...
Processing 2862_or_mb1_mdr_rdr_dp_mb2.xml...
Processing 1018_or_mb1_mdr_rdr_dp_mb2.xml...
Processing 428_or_mb1_mdr_rdr_dp_mb2.xml...
Processing 939_or_mb1_mdr_rdr_dp_mb2.xml...
Processing 709_or_mb1_mdr_rdr_dp_mb2.xml...
Processing 978_or_mb1_mdr_rdr_dp_mb2.xml...
Processing 868_or_mb1_mdr_rdr_dp_mb2.xml...
Processing 778_or_mb1_mdr_rdr_dp_mb2.xml...
Processing 895_or_mb1_mdr_rdr_dp_mb2.xml...
Processing 2774_or_mb1_mdr_rdr_dp_mb2.xml...
Processing 459_or_mb1_mdr_rdr_dp_mb2.xml...
Processing 262_or_mb1_mdr_rdr_dp_mb2.xml...
Processing 761_or_mb1_mdr_rdr_dp_mb2.xml...
Processing 163_or_mb1_mdr_rdr_dp_mb2.xml...
Processing 997_or_mb1_mdr_rdr_dp_mb2.xml...
Processing 100_or_mb1_mdr_rdr_dp_mb2.xml...
Processing 1252_or_mb1_mdr_rdr_dp_mb2.xml...
Processing 2751_or_mb1_mdr_rdr_dp_mb2.xml...
Processing 1114_or_mb1_md

#### FIX

In [61]:
loaded_model_save = {}
for file in os.listdir(save_dir):
    if not file.endswith(('.xml', '.sbml')):
        continue
        
    model = read_sbml_model(os.path.join(save_dir, file))
    model_id = model.id
    loaded_model_save[model_id] = model

In [52]:
def overwrite_reaction(model, rxn_id, new_rxn_dict):
    if rxn_id in model.reactions:
        try:
            rxn = model.reactions.get_by_id(rxn_id)
            old_metabolites = {met.id: coeff for met, coeff in rxn.metabolites.items()}
            rxn.subtract_metabolites(rxn.metabolites)
            rxn.add_metabolites(new_rxn_dict)
            new_metabolites = {met.id: coeff for met, coeff in rxn.metabolites.items()}
        except KeyError:
            print(f"{model} does not contain one of the metabolites")

In [62]:
for model_id, model in loaded_model_save.items():
    #overwrite_reaction(model, "PSP", {"h2o_c": -1.0, "pser__L_c": -1.0, "pi_c": 1.0, "ser__L_c": 1.0}) #was wrong reaction before
    if "HDAD" in model.reactions:
        print(model_id, model.reactions.HDAD)

m_2872_ HDAD: 49dsha_c + h2o_c --> hhd_c + hip_c
m_1357_ HDAD: 49dsha_c + h2o_c --> hhd_c + hip_c
m_397_ HDAD: 49dsha_c + h2o_c --> hhd_c + hip_c
m_1018_ HDAD: 49dsha_c + h2o_c --> hhd_c + hip_c
m_978_ HDAD: 49dsha_c + h2o_c --> hhd_c + hip_c
m_2774_ HDAD: 49dsha_c + h2o_c --> hhd_c + hip_c
m_459_ HDAD: 49dsha_c + h2o_c --> hhd_c + hip_c
m_997_ HDAD: 49dsha_c + h2o_c --> hhd_c + hip_c
m_352_ HDAD: 49dsha_c + h2o_c --> hhd_c + hip_c
m_638_ HDAD: 49dsha_c + h2o_c --> hhd_c + hip_c
m_793_ HDAD: 49dsha_c + h2o_c --> hhd_c + hip_c
m_364_ HDAD: 49dsha_c + h2o_c --> hhd_c + hip_c
m_1432_ HDAD: 49dsha_c + h2o_c --> hhd_c + hip_c
m_161_ HDAD: 49dsha_c + h2o_c --> hhd_c + hip_c
m_230_ HDAD: 49dsha_c + h2o_c --> hhd_c + hip_c
m_1208_ HDAD: 49dsha_c + h2o_c --> hhd_c + hip_c
m_1174_ HDAD: 49dsha_c + h2o_c --> hhd_c + hip_c
m_504_ HDAD: 49dsha_c + h2o_c --> hhd_c + hip_c
m_1234_ HDAD: 49dsha_c + h2o_c --> hhd_c + hip_c
m_790_ HDAD: 49dsha_c + h2o_c --> hhd_c + hip_c
m_946_ HDAD: 49dsha_c + h2o_c --

In [64]:
m_2872 = loaded_model_save["m_2872_"]

In [67]:
m_2872.metabolites.get_by_id("5ohhip_c")

Metabolite identifier,5ohhip_c
Name,"3-[(3aS,4S,7aS)-7a-Methyl-1-hydroxy-5-oxo-..."
Memory address,0x7e78984920e0
Formula,C13H19O4
Compartment,C_c
In 2 reaction(s),"IPDAB, HIPA"


In [65]:
for r in m_2872.metabolites.hip_c.reactions:
    print(r)

FADD3: atp_c + coa_c + hip_c --> amp_c + hipcoa_c + ppi_c
HDAD: 49dsha_c + h2o_c --> hhd_c + hip_c
HIPA: hip_c + nadh_c --> 5ohhip_c + nad_c


In [63]:
for model_id, model in loaded_model_save.items():
    if "hip_c" in model.metabolites:
        print(model_id, model.metabolites.get_by_id("hip_c").reactions)


m_2872_ frozenset({<Reaction FADD3 at 0x7e789c367610>, <Reaction HDAD at 0x7e7895ded120>, <Reaction HIPA at 0x7e7895deed70>})
m_1357_ frozenset({<Reaction FADD3 at 0x7e7898c21e10>, <Reaction HDAD at 0x7e78965789a0>})
m_397_ frozenset({<Reaction FADD3 at 0x7e78931d3880>, <Reaction HDAD at 0x7e7898fe1030>})
m_1018_ frozenset({<Reaction FADD3 at 0x7e78909e2200>, <Reaction HDAD at 0x7e7891e9b400>})
m_978_ frozenset({<Reaction HDAD at 0x7e78e2609f00>, <Reaction FADD3 at 0x7e78e043b9a0>, <Reaction HIPA at 0x7e78e27c8970>})
m_2774_ frozenset({<Reaction HIPA at 0x7e788f69b220>, <Reaction HDAD at 0x7e78e1c15030>})
m_459_ frozenset({<Reaction HDAD at 0x7e78b51c8850>, <Reaction FADD3 at 0x7e78ec5ccbe0>})
m_997_ frozenset({<Reaction HDAD at 0x7e78c17cf820>, <Reaction FADD3 at 0x7e78b7c43eb0>})
m_352_ frozenset({<Reaction HIPA at 0x7e78935e2b00>, <Reaction HDAD at 0x7e78935e1270>})
m_638_ frozenset({<Reaction HDAD at 0x7e78d41c7880>, <Reaction HIPA at 0x7e78d1b40dc0>})
m_793_ frozenset({<Reaction F

In [29]:
m_796 = loaded_model_save["m_796_"]

In [ ]:
for r in m_796.metabolites.hip_c.reactions:
    print(r)

GLYCS_I: gthrd_c + mthgxl_c --> lgt__S_c
LALDO: gthrd_c + lald__D_c + nad_c <=> h_c + lgt__S_c + nadh_c
GLYCS_II: h2o_c + lgt__S_c --> gthrd_c + h_c + lac__L_c
GLYOX: h2o_c + lgt__S_c --> gthrd_c + h_c + lac__D_c
GLYOX_1: lgt__S_c <=> gthrd_c + mthgxl_c


In [30]:
m_796.reactions.GLYOX_1

Reaction identifier,GLYOX_1
Name,Glyoxalase I; Ni-dependent
Memory address,0x7e78e93537c0
Stoichiometry,lgt__S_c <=> gthrd_c + mthgxl_c (R)-S-Lactoylglutathione <=> Reduced glutathione + Methylglyoxal
GPR,
Lower bound,-1000.0
Upper bound,1000.0


In [31]:
m_796.reactions.GLYCS_I

Reaction identifier,GLYCS_I
Name,
Memory address,0x7e78e9350700
Stoichiometry,gthrd_c + mthgxl_c --> lgt__S_c Reduced glutathione + Methylglyoxal --> (R)-S-Lactoylglutathione
GPR,CPMLCC_01640 or CPMLCC_00643 or CPMLCC_01460
Lower bound,0.0
Upper bound,1000.0


In [32]:
m_868 = loaded_model_save["m_868_"]

In [35]:
m_868.reactions.LGTHL

Reaction identifier,LGTHL
Name,Lactoylglutathione lyase
Memory address,0x7e78e0e575e0
Stoichiometry,gthrd_c + mthgxl_c --> lgt__S_c Reduced glutathione + Methylglyoxal --> (R)-S-Lactoylglutathione
GPR,NCJNGA_00122 or NCJNGA_02705
Lower bound,0.0
Upper bound,1000.0
